# Stock History Loader

This notebook pulls stock history into the local AlphaLens bronze/silver/gold stores.
It uses the notebook-oriented loader in `fg.pipelines.historical_loader`, including the SEC canonicalization pass needed for live `companyfacts` payloads.

The default universe below is the top 100 unique S&P 500 companies by index weight, based on Slickcharts data accessed on March 7, 2026. Alphabet is represented once as `GOOGL`, and Berkshire is normalized to `BRK-B` for SEC/Yahoo compatibility.

For bulk historical loads, forward estimates are disabled by default so the notebook can collect SEC and Yahoo history without depending on paid FMP coverage.

In [ ]:
import pandas as pd

from fg.pipelines import HistoricalDataLoader
from fg.settings import get_settings

settings = get_settings()
loader = HistoricalDataLoader(settings)
settings.data_dirs

In [ ]:
TICKERS = [
    "NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "META", "AVGO", "TSLA", "BRK-B", "WMT",
    "LLY", "JPM", "XOM", "V", "JNJ", "MA", "COST", "ORCL", "NFLX", "MU",
    "ABBV", "CVX", "PLTR", "PG", "HD", "BAC", "GE", "KO", "CAT", "AMD",
    "CSCO", "MRK", "RTX", "PM", "UNH", "AMAT", "MS", "LRCX", "WFC", "GS",
    "TMUS", "IBM", "MCD", "LIN", "PEP", "INTC", "VZ", "GEV", "AXP", "T",
    "AMGN", "ABT", "NEE", "CRM", "TMO", "C", "BA", "DIS", "GILD", "TJX",
    "KLAC", "TXN", "ISRG", "APP", "SCHW", "ANET", "APH", "DE", "UBER", "LMT",
    "ADI", "PFE", "UNP", "HON", "BLK", "QCOM", "BKNG", "COP", "WELL", "LOW",
    "SYK", "DHR", "SPGI", "ETN", "PANW", "INTU", "ACN", "NOW", "CB", "NEM",
    "PLD", "PGR", "BMY", "HCA", "COF", "MDT", "PH", "ADBE", "VRTX", "CEG",
]
WATCHLIST_NAME = None
USE_FIXTURES = False
INCLUDE_ESTIMATES = False
BUILD_GOLD = False
LOOKBACK_YEARS = 20
PE_METHOD = "static_15"
SHOW_ESTIMATES = False
MANUAL_PE = None
CONTINUE_ON_ERROR = True
BATCH_SIZE = 25

In [ ]:
if WATCHLIST_NAME:
    TICKERS = [
        str(ticker).upper().replace(".", "-")
        for ticker in settings.watchlists_config.get("watchlists", {}).get(WATCHLIST_NAME, [])
    ]
else:
    TICKERS = [str(ticker).upper().replace(".", "-") for ticker in TICKERS]

summary_frames = []
batch_count = (len(TICKERS) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_index, start in enumerate(range(0, len(TICKERS), BATCH_SIZE), start=1):
    batch = TICKERS[start:start + BATCH_SIZE]
    print(f"Loading batch {batch_index}/{batch_count}: {batch[0]} -> {batch[-1]}")
    batch_df = loader.load_tickers(
        batch,
        fixture_mode=USE_FIXTURES,
        include_estimates=INCLUDE_ESTIMATES,
        build_gold=BUILD_GOLD,
        lookback_years=LOOKBACK_YEARS,
        pe_method=PE_METHOD,
        show_estimates=SHOW_ESTIMATES,
        manual_pe=MANUAL_PE,
        continue_on_error=CONTINUE_ON_ERROR,
    )
    summary_frames.append(batch_df)

summary_df = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
summary_df = summary_df.sort_values(["status", "ticker"]).reset_index(drop=True)
summary_df

In [ ]:
loader.table_inventory()

In [ ]:
created_views = loader.register_duckdb_views()
created_views

In [ ]:
loader.query_duckdb(
    """
    SELECT ticker, company_key, issuer_name, last_sec_pull_at, last_yahoo_pull_at, last_fmp_pull_at
    FROM v_dim_company
    ORDER BY ticker
    """
)

In [ ]:
loader.query_duckdb(
    """
    SELECT company_key, metric_code, COUNT(*) AS row_count
    FROM v_fundamental_annual
    GROUP BY company_key, metric_code
    ORDER BY company_key, metric_code
    """
)